# 📦 02 — Vehicle Dataset Exploration
## SmartMine Vision AI · Module 2: Vehicle Detection

---

### Objectives
1. Count **images and annotations** per source and split.
2. Analyse **class names and distribution** across all four sources.
3. Visualise **bounding box geometry** (size, aspect ratio, position).
4. Identify **overlapping class semantics** across sources (prep for merging).

---

> **Prerequisite:** Run `01_dataset_download.ipynb` first to ensure all four
> datasets are present under `datasets/raw/vehicles/`.

## 1. Setup

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_cwd = Path().resolve()
PROJECT_ROOT = _cwd
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

VEHICLES_DIR = PROJECT_ROOT / "datasets" / "raw" / "vehicles"
print(f"Project root : {PROJECT_ROOT}")
print(f"Vehicles dir : {VEHICLES_DIR}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from collections import Counter

SOURCES = {
    "construction_vehicles": {
        "workspace": "0925",
        "project":   "construction-vehicle-inspection",
        "version":   1,
        "url": "https://universe.roboflow.com/0925/construction-vehicle-inspection",
    },
    "mining_area_detection": {
        "workspace": "septiana-s-workspace",
        "project":   "mining-area-vehicle-detection",
        "version":   1,
        "url": "https://universe.roboflow.com/septiana-s-workspace/mining-area-vehicle-detection",
    },
    "riskalert": {
        "workspace": "personal-q02wc",
        "project":   "riskalert-mining",
        "version":   1,
        "url": "https://universe.roboflow.com/personal-q02wc/riskalert-mining",
    },
    "riskalertai": {
        "workspace": "personal-q02wc",
        "project":   "riskalertai-mining",
        "version":   10,
        "url": "https://universe.roboflow.com/personal-q02wc/riskalertai-mining",
    },
}

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
print("Imports OK.")

## 2. Data Availability Guard

In [ ]:
def images_in(path: Path) -> int:
    return sum(1 for _ in path.rglob("*.jpg")) + sum(1 for _ in path.rglob("*.png"))

available = {
    alias: images_in(VEHICLES_DIR / alias) > 0
    for alias in SOURCES
}

DATA_AVAILABLE = all(available.values())

print("Availability:")
for alias, ok in available.items():
    print(f"  {alias:<30} {'OK' if ok else 'MISSING'}")
print()
if not DATA_AVAILABLE:
    missing = [a for a, ok in available.items() if not ok]
    print(f"[INFO] Missing datasets: {missing}")
    print("Run 01_dataset_download.ipynb to download them.")
    print("Analysis cells below will be skipped.")

## 3. Image and Annotation Counts per Source

In [ ]:
if DATA_AVAILABLE:
    rows = []
    for alias in SOURCES:
        src_dir = VEHICLES_DIR / alias

        # Roboflow downloads use train/valid/test structure
        for split in ("train", "valid", "test"):
            img_dir = src_dir / split / "images"
            lbl_dir = src_dir / split / "labels"
            if not img_dir.exists():
                # some exports use different paths; try direct
                img_dir = src_dir / "images"
                lbl_dir = src_dir / "labels"
            if not img_dir.exists():
                continue

            imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
            anns = sum(
                len([l for l in (lbl_dir / (p.stem + ".txt")).read_text().splitlines()
                     if l.strip()])
                for p in imgs
                if (lbl_dir / (p.stem + ".txt")).exists()
            )
            rows.append({
                "source": alias,
                "split": split,
                "images": len(imgs),
                "annotations": anns,
            })

    count_df = pd.DataFrame(rows)

    print("IMAGE & ANNOTATION COUNTS")
    print("=" * 60)
    print(count_df.to_string(index=False))
    print()

    totals = count_df.groupby("source")[["images", "annotations"]].sum()
    print("TOTALS PER SOURCE")
    print("-" * 40)
    print(totals.to_string())
    print(f"\nGRAND TOTAL: {totals['images'].sum():,} images  {totals['annotations'].sum():,} annotations")
else:
    print("[SKIP] Data not available.")

In [ ]:
if DATA_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    pivot = count_df.pivot_table(index="source", columns="split",
                                  values="images", fill_value=0)
    pivot.plot(kind="bar", ax=axes[0], colormap="Set2", edgecolor="white")
    axes[0].set_title("Images per Source & Split")
    axes[0].set_xlabel("")
    axes[0].tick_params(axis="x", rotation=30)

    totals.reset_index().plot.bar(x="source", y="annotations", ax=axes[1],
                                   color="#7b68ee", edgecolor="white")
    axes[1].set_title("Total Annotations per Source")
    axes[1].set_xlabel("")
    axes[1].tick_params(axis="x", rotation=30)

    plt.suptitle("Vehicle Dataset Overview", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 4. Class Names per Source

In [ ]:
if DATA_AVAILABLE:
    source_classes = {}
    for alias in SOURCES:
        src_dir = VEHICLES_DIR / alias
        yaml_files = list(src_dir.rglob("*.yaml"))
        if not yaml_files:
            print(f"  [WARN] No data.yaml found for {alias}")
            continue
        with open(yaml_files[0]) as f:
            cfg = yaml.safe_load(f)
        names = cfg.get("names", {})
        if isinstance(names, dict):
            names = list(names.values())
        source_classes[alias] = names
        print(f"  {alias} ({len(names)} classes): {names[:10]}{'...' if len(names)>10 else ''}")

    print()

    # Show classes that appear in multiple sources (potential merge candidates)
    all_classes = Counter()
    for names in source_classes.values():
        all_classes.update(n.lower() for n in names)

    shared = {cls: cnt for cls, cnt in all_classes.items() if cnt > 1}
    print(f"Classes appearing in 2+ sources ({len(shared)}):")
    for cls, cnt in sorted(shared.items(), key=lambda x: -x[1]):
        print(f"  {cls:<35} in {cnt} sources")
else:
    print("[SKIP] Data not available.")

## 5. Bounding Box Geometry Analysis

In [ ]:
if DATA_AVAILABLE:
    all_boxes = {alias: [] for alias in SOURCES}

    for alias in SOURCES:
        src_dir = VEHICLES_DIR / alias
        label_files = list(src_dir.rglob("*.txt"))
        for txt in label_files[:300]:  # sample first 300 per source
            for line in txt.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) == 5:
                    try:
                        all_boxes[alias].append([float(p) for p in parts[1:]])
                    except ValueError:
                        pass

    fig, axes = plt.subplots(len(SOURCES), 3, figsize=(15, 4 * len(SOURCES)))
    if len(SOURCES) == 1:
        axes = [axes]

    for row_idx, alias in enumerate(SOURCES):
        boxes = np.array(all_boxes[alias]) if all_boxes[alias] else np.empty((0, 4))
        if len(boxes) == 0:
            for ax in axes[row_idx]:
                ax.text(0.5, 0.5, "no data", ha="center", va="center")
                ax.set_title(alias)
            continue

        cx, cy, w, h = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]

        axes[row_idx][0].hist(w, bins=40, alpha=0.7, color="#4CAF50", label="width")
        axes[row_idx][0].hist(h, bins=40, alpha=0.7, color="#2196F3", label="height")
        axes[row_idx][0].set_title(f"{alias} — BBox size")
        axes[row_idx][0].legend(fontsize=8)

        axes[row_idx][1].hist2d(cx, cy, bins=30, cmap="YlOrRd")
        axes[row_idx][1].set_title(f"{alias} — Position heatmap")
        axes[row_idx][1].invert_yaxis()

        ar = w / np.maximum(h, 1e-6)
        axes[row_idx][2].hist(ar, bins=40, alpha=0.8, color="#FF9800")
        axes[row_idx][2].axvline(1.0, color="red", ls="--", alpha=0.6)
        axes[row_idx][2].set_title(f"{alias} — Aspect ratio")

    plt.suptitle("BBox Geometry per Source", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("[SKIP] Data not available.")

## 6. Class Overlap Assessment

Before merging these sources with `scripts/merge_datasets.py`, the class maps
need to be harmonised. The table below captures the key vehicle classes found
and their semantic equivalents across sources:

| Canonical class | construction_vehicles | mining_area_detection | riskalert | riskalertai |
|---|---|---|---|---|
| `camion` / truck | ✓ | ✓ | ✓ | ✓ |
| `excavadora` | — | ✓ | ✓ | ✓ |
| `volquete` | — | ✓ | ✓ | ✓ |
| `cargador_frontal` | — | — | ✓ | ✓ |
| `motoniveladora` | — | — | ✓ | ✓ |
| `cisterna_agua` | — | — | ✓ | ✓ |
| `car` / `auto` | ✓ | ✓ | — | — |

> This table is populated by running this notebook; update it after the
> sources download. It feeds the class map in `scripts/merge_datasets.py`.

## 7. Next Steps

1. Review class overlap table above and update `scripts/merge_datasets.py`
   class map to include the two new sources (`construction_vehicles`,
   `mining_area_detection`, `riskalertai`).
2. Run `python scripts/merge_datasets.py` to rebuild `datasets/merged/smartmine_v1/`.
3. Proceed to `notebooks/01_ppe_detection/03_training_yolo.ipynb` with the
   updated corpus.